# Chapter 3.7: Generative Retrieval

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** DSI (Differentiable Search Index): the document-as-token paradigm
2. **Explain** TIGER's approach: RQ-VAE semantic IDs + autoregressive generation
3. **Describe** NCI's prefix-aware encoding for improved generative retrieval
4. **Implement** GENRE-style constrained beam search for entity retrieval
5. **Understand** TDM (Tree-based Deep Model) and DR (Deep Retrieval) architectures
6. **Compare** index-based vs model-based retrieval paradigms
7. **Build** a mini generative retrieval system with autoregressive decoding

## Prerequisites

- Semantic IDs and RQ-VAE (Chapter 3.6)
- Transformer and autoregressive generation basics
- Two-tower retrieval (Chapter 3.1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part3/chapter_3.7_generative_retrieval.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part3/chapter_3.7_generative_retrieval.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print(f"PyTorch version: {torch.__version__}")

## 1. The Generative Retrieval Paradigm

Traditional retrieval: **encode → index → search** (retrieve from an external index)

Generative retrieval: **encode → generate** (the model itself memorizes the index)

Instead of computing embeddings and searching an ANN index, generative retrieval models directly **generate** item identifiers token by token.

$$P(\text{item} | \text{user context}) = \prod_{l=1}^{L} P(c_l | c_{<l}, \text{user context})$$

where $(c_1, c_2, \ldots, c_L)$ is the item's semantic ID.

> **💡 Concept:** Generative retrieval eliminates the ANN index entirely. The model's parameters serve as the index. This unifies the encoding and retrieval steps into a single model, potentially allowing end-to-end optimization.

In [ ]:
# Illustrate the paradigm shift
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Traditional retrieval flow
steps_trad = ['User\nFeatures', 'User\nEncoder', 'User\nEmbedding', 'ANN\nIndex', 'Top-K\nItems']
for i, step in enumerate(steps_trad):
    color = '#3498db' if i < 3 else '#e74c3c' if i == 3 else '#2ecc71'
    axes[0].add_patch(plt.Rectangle((i * 1.8, 0.2), 1.5, 0.6, 
                      facecolor=color, alpha=0.3, edgecolor=color))
    axes[0].text(i * 1.8 + 0.75, 0.5, step, ha='center', va='center', fontsize=9)
    if i < len(steps_trad) - 1:
        axes[0].annotate('', xy=((i+1) * 1.8, 0.5), xytext=(i * 1.8 + 1.5, 0.5),
                        arrowprops=dict(arrowstyle='->', color='gray'))
axes[0].set_xlim(-0.2, 9.5)
axes[0].set_ylim(0, 1)
axes[0].set_title('Traditional: Encode + Index + Search', fontsize=13)
axes[0].axis('off')

# Generative retrieval flow
steps_gen = ['User\nFeatures', 'Encoder', 'Generate\nc1', 'Generate\nc2', 'Generate\nc3', 'Item ID\n(c1,c2,c3)']
for i, step in enumerate(steps_gen):
    color = '#3498db' if i < 2 else '#9b59b6' if i < 5 else '#2ecc71'
    axes[1].add_patch(plt.Rectangle((i * 1.5, 0.2), 1.3, 0.6,
                      facecolor=color, alpha=0.3, edgecolor=color))
    axes[1].text(i * 1.5 + 0.65, 0.5, step, ha='center', va='center', fontsize=9)
    if i < len(steps_gen) - 1:
        axes[1].annotate('', xy=((i+1) * 1.5, 0.5), xytext=(i * 1.5 + 1.3, 0.5),
                        arrowprops=dict(arrowstyle='->', color='gray'))
axes[1].set_xlim(-0.2, 9.5)
axes[1].set_ylim(0, 1)
axes[1].set_title('Generative: Encode + Generate Token-by-Token', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. DSI: Differentiable Search Index (Google, 2022)

DSI (Tay et al., 2022, Google) was the first major generative retrieval work. The model memorizes document IDs by training on:

1. **Indexing task**: Given document content, predict its document ID
2. **Retrieval task**: Given query, predict relevant document ID

Key design choices for document IDs:
- **Unstructured atomic IDs**: Just assign random integers (poor)
- **Naively structured IDs**: Assign integers by document order (poor)
- **Hierarchical semantic IDs**: Cluster documents, assign hierarchical codes (best)

> **⚠️ Common Pitfall:** DSI with atomic IDs performs poorly because random integers carry no structure. The model must memorize arbitrary mappings. Semantic IDs dramatically improve performance by making the ID space meaningful.

In [ ]:
class DSIModel(nn.Module):
    """Simplified Differentiable Search Index (Tay et al., 2022).
    
    Uses a Transformer encoder-decoder architecture.
    Encoder: processes query/document
    Decoder: generates document ID tokens autoregressively
    """
    
    def __init__(self, input_dim: int, vocab_size: int = 64, 
                 num_id_tokens: int = 3, d_model: int = 64, 
                 nhead: int = 2, num_layers: int = 2):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_id_tokens = num_id_tokens
        self.d_model = d_model
        
        # Encoder: process input (query or document)
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Decoder: generate ID tokens
        self.id_embedding = nn.Embedding(vocab_size + 2, d_model)  # +2 for BOS, PAD
        self.bos_token = vocab_size  # BOS token ID
        self.pos_embedding = nn.Embedding(num_id_tokens + 1, d_model)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input features."""
        h = self.input_proj(x).unsqueeze(1)  # (B, 1, D)
        return self.encoder(h)  # (B, 1, D)
    
    def forward(self, x: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
        """Teacher-forced training.
        
        Args:
            x: (B, input_dim) input features
            target_ids: (B, L) target semantic IDs
        
        Returns:
            logits: (B, L, vocab_size)
        """
        B, L = target_ids.shape
        memory = self.encode(x)  # (B, 1, D)
        
        # Prepare decoder input: BOS + target_ids[:-1]
        bos = torch.full((B, 1), self.bos_token, dtype=torch.long, device=x.device)
        decoder_input = torch.cat([bos, target_ids[:, :-1]], dim=1)  # (B, L)
        
        positions = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)
        tgt = self.id_embedding(decoder_input) + self.pos_embedding(positions)
        
        # Causal mask for decoder
        causal_mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        
        output = self.decoder(tgt, memory, tgt_mask=causal_mask)  # (B, L, D)
        logits = self.output_proj(output)  # (B, L, vocab_size)
        return logits
    
    @torch.no_grad()
    def generate(self, x: torch.Tensor, beam_size: int = 1) -> torch.Tensor:
        """Generate semantic IDs autoregressively."""
        B = x.size(0)
        memory = self.encode(x)
        
        if beam_size == 1:
            # Greedy decoding
            generated = torch.full((B, 1), self.bos_token, dtype=torch.long, device=x.device)
            
            for step in range(self.num_id_tokens):
                positions = torch.arange(generated.size(1), device=x.device).unsqueeze(0).expand(B, -1)
                tgt = self.id_embedding(generated) + self.pos_embedding(positions)
                causal_mask = torch.triu(
                    torch.ones(tgt.size(1), tgt.size(1), device=x.device), diagonal=1
                ).bool()
                output = self.decoder(tgt, memory, tgt_mask=causal_mask)
                logits = self.output_proj(output[:, -1])  # (B, vocab_size)
                next_token = logits.argmax(dim=-1, keepdim=True)  # (B, 1)
                generated = torch.cat([generated, next_token], dim=1)
            
            return generated[:, 1:]  # Remove BOS, return (B, L)
        else:
            return self._beam_search(memory, x.device, beam_size)
    
    def _beam_search(self, memory, device, beam_size):
        """Simple beam search for a single example."""
        B = memory.size(0)
        assert B == 1, "Beam search implemented for batch_size=1"
        
        # Initialize beams: (beam_size, seq_len)
        beams = [(0.0, [self.bos_token])]  # (score, token_list)
        
        for step in range(self.num_id_tokens):
            all_candidates = []
            for score, tokens in beams:
                token_tensor = torch.tensor([tokens], dtype=torch.long, device=device)
                positions = torch.arange(len(tokens), device=device).unsqueeze(0)
                tgt = self.id_embedding(token_tensor) + self.pos_embedding(positions)
                causal_mask = torch.triu(
                    torch.ones(len(tokens), len(tokens), device=device), diagonal=1
                ).bool()
                output = self.decoder(tgt, memory, tgt_mask=causal_mask)
                logits = self.output_proj(output[:, -1])  # (1, vocab_size)
                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)
                
                top_k_probs, top_k_ids = log_probs.topk(beam_size)
                for k in range(beam_size):
                    new_score = score + top_k_probs[k].item()
                    new_tokens = tokens + [top_k_ids[k].item()]
                    all_candidates.append((new_score, new_tokens))
            
            # Keep top beam_size candidates
            beams = sorted(all_candidates, key=lambda x: -x[0])[:beam_size]
        
        # Return all beam results
        results = torch.tensor([b[1][1:] for b in beams], device=device)  # Remove BOS
        scores = torch.tensor([b[0] for b in beams], device=device)
        return results, scores

print("DSI model created.")

## 3. TIGER: Semantic IDs + Autoregressive Generation (Google, 2023)

TIGER (Rajput et al., 2023, Google) combines RQ-VAE semantic IDs with a Transformer-based generative model:

1. **Step 1**: Train RQ-VAE to assign semantic IDs to all items
2. **Step 2**: Train a sequence-to-sequence model: user history → item semantic ID

The user's interaction history $[i_1, i_2, \ldots, i_t]$ is mapped to the semantic ID of the next item $(c_1, c_2, c_3)$.

### Key Design Choices
- **Semantic ID quality** directly impacts retrieval quality
- **Beam search** at inference to retrieve multiple candidates
- **Constrained decoding**: only generate valid (existing) semantic IDs

> **🔑 Pro Tip:** TIGER showed that semantic IDs are crucial — using atomic IDs with the same generative framework performs much worse. The structure in semantic IDs helps the model generalize.

In [ ]:
# Generate synthetic data and semantic IDs
NUM_ITEMS = 2000
NUM_USERS = 300
EMBED_DIM = 32
NUM_CODES = 32  # Per level
NUM_LEVELS = 3
NUM_CATEGORIES = 8

# Create items with category structure
item_categories = np.random.randint(0, NUM_CATEGORIES, NUM_ITEMS)
category_centers = np.random.randn(NUM_CATEGORIES, EMBED_DIM).astype(np.float32) * 2
item_features = np.array([
    category_centers[cat] + np.random.randn(EMBED_DIM).astype(np.float32) * 0.3
    for cat in item_categories
])

# Simple semantic ID assignment (simulate RQ-VAE output)
# Level 1: based on category
from sklearn.cluster import KMeans

# Use k-means as a simple proxy for VQ quantization
semantic_ids = np.zeros((NUM_ITEMS, NUM_LEVELS), dtype=np.int64)

# Level 1: coarse clustering
kmeans1 = KMeans(n_clusters=NUM_CODES, random_state=42, n_init=3)
semantic_ids[:, 0] = kmeans1.fit_predict(item_features)

# Level 2: sub-clustering within level 1
residuals1 = item_features - kmeans1.cluster_centers_[semantic_ids[:, 0]]
kmeans2 = KMeans(n_clusters=NUM_CODES, random_state=42, n_init=3)
semantic_ids[:, 1] = kmeans2.fit_predict(residuals1)

# Level 3: sub-clustering within level 2
residuals2 = residuals1 - kmeans2.cluster_centers_[semantic_ids[:, 1]]
kmeans3 = KMeans(n_clusters=NUM_CODES, random_state=42, n_init=3)
semantic_ids[:, 2] = kmeans3.fit_predict(residuals2)

semantic_ids_tensor = torch.tensor(semantic_ids, dtype=torch.long)

# Build valid ID trie for constrained decoding
valid_ids = set()
for sid in semantic_ids:
    valid_ids.add(tuple(sid.tolist()))

print(f"Items: {NUM_ITEMS}, Semantic ID levels: {NUM_LEVELS}, Codes per level: {NUM_CODES}")
print(f"Unique valid IDs: {len(valid_ids)}")
print(f"Possible IDs: {NUM_CODES**NUM_LEVELS}")
print(f"Example IDs: {semantic_ids[:5].tolist()}")

In [ ]:
# Generate user sequences
user_sequences = {}
for u in range(NUM_USERS):
    # User has 2-3 preferred categories
    n_prefs = np.random.choice([2, 3])
    prefs = np.random.choice(NUM_CATEGORIES, n_prefs, replace=False)
    
    seq_len = np.random.randint(10, 30)
    sequence = []
    for _ in range(seq_len):
        cat = np.random.choice(prefs) if np.random.random() < 0.8 else np.random.randint(0, NUM_CATEGORIES)
        items_in_cat = np.where(item_categories == cat)[0]
        item = np.random.choice(items_in_cat)
        sequence.append(item)
    user_sequences[u] = sequence

# Create training data: (history_features, target_semantic_id)
from torch.utils.data import Dataset, DataLoader

class GenerativeRetrievalDataset(Dataset):
    def __init__(self, user_sequences, item_features, semantic_ids, max_hist=20):
        self.samples = []
        self.max_hist = max_hist
        
        for u, seq in user_sequences.items():
            for t in range(1, len(seq)):
                hist = seq[max(0, t-max_hist):t]
                target = seq[t]
                # Average pool history features
                hist_feat = item_features[hist].mean(axis=0)
                target_sid = semantic_ids[target]
                self.samples.append((hist_feat, target_sid))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        feat, sid = self.samples[idx]
        return {
            'features': torch.tensor(feat, dtype=torch.float),
            'semantic_id': torch.tensor(sid, dtype=torch.long)
        }

train_data = GenerativeRetrievalDataset(user_sequences, item_features, semantic_ids)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
print(f"Training samples: {len(train_data)}")

In [ ]:
# Train DSI/TIGER-style model
model = DSIModel(EMBED_DIM, vocab_size=NUM_CODES, num_id_tokens=NUM_LEVELS, 
                 d_model=64, nhead=2, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
model.train()

for epoch in range(30):
    epoch_loss = 0
    n_batch = 0
    for batch in train_loader:
        logits = model(batch['features'], batch['semantic_id'])  # (B, L, V)
        
        loss = F.cross_entropy(
            logits.reshape(-1, NUM_CODES),
            batch['semantic_id'].reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batch += 1
    
    avg_loss = epoch_loss / n_batch
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses, 'o-', color='purple', linewidth=2, markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.title('Generative Retrieval Training Loss', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate: Generate semantic IDs and check correctness
model.eval()

# Build reverse mapping: semantic_id -> item_ids
sid_to_items = defaultdict(list)
for item_id, sid in enumerate(semantic_ids):
    sid_to_items[tuple(sid.tolist())].append(item_id)

# Test on some examples
n_correct_exact = 0
n_correct_category = 0
n_total = 0

for u in range(min(100, NUM_USERS)):
    seq = user_sequences[u]
    if len(seq) < 3:
        continue
    
    hist = seq[:-1][-20:]
    target = seq[-1]
    target_cat = item_categories[target]
    target_sid = tuple(semantic_ids[target].tolist())
    
    hist_feat = torch.tensor(item_features[hist].mean(axis=0), dtype=torch.float).unsqueeze(0)
    
    generated_ids = model.generate(hist_feat, beam_size=1)
    gen_sid = tuple(generated_ids[0].tolist())
    
    # Check if generated ID is valid and maps to correct category
    if gen_sid == target_sid:
        n_correct_exact += 1
    
    if gen_sid in sid_to_items:
        retrieved_items = sid_to_items[gen_sid]
        retrieved_cats = [item_categories[i] for i in retrieved_items]
        if target_cat in retrieved_cats:
            n_correct_category += 1
    
    n_total += 1

print(f"\nGenerative Retrieval Evaluation ({n_total} users):")
print(f"  Exact ID match:    {n_correct_exact/n_total:.4f}")
print(f"  Category match:    {n_correct_category/n_total:.4f}")

## 4. Constrained Beam Search (GENRE-style)

GENRE (De Cao et al., 2021, Meta) uses **constrained beam search** to only generate valid entity names. We adapt this for semantic IDs using a **trie** (prefix tree) of valid IDs.

At each decoding step, we mask out tokens that would lead to invalid (non-existent) ID prefixes.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.item_ids = []  # Items with this exact ID

class SemanticIDTrie:
    """Prefix tree for valid semantic IDs."""
    
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, semantic_id: tuple, item_id: int):
        node = self.root
        for code in semantic_id:
            if code not in node.children:
                node.children[code] = TrieNode()
            node = node.children[code]
        node.is_end = True
        node.item_ids.append(item_id)
    
    def get_valid_continuations(self, prefix: list) -> set:
        """Get valid next tokens given a prefix."""
        node = self.root
        for code in prefix:
            if code not in node.children:
                return set()
            node = node.children[code]
        return set(node.children.keys())
    
    def lookup(self, semantic_id: tuple) -> list:
        """Get item IDs for a complete semantic ID."""
        node = self.root
        for code in semantic_id:
            if code not in node.children:
                return []
            node = node.children[code]
        return node.item_ids if node.is_end else []

# Build trie
trie = SemanticIDTrie()
for item_id, sid in enumerate(semantic_ids):
    trie.insert(tuple(sid.tolist()), item_id)

# Constrained beam search
def constrained_beam_search(model, x, trie, beam_size=5):
    """Beam search constrained to valid semantic IDs."""
    memory = model.encode(x)
    
    beams = [(0.0, [])]  # (score, generated_tokens)
    
    for step in range(model.num_id_tokens):
        all_candidates = []
        
        for score, tokens in beams:
            # Get valid continuations from trie
            valid_tokens = trie.get_valid_continuations(tokens)
            if not valid_tokens:
                continue
            
            # Compute logits
            full_tokens = [model.bos_token] + tokens
            token_tensor = torch.tensor([full_tokens], dtype=torch.long, device=x.device)
            positions = torch.arange(len(full_tokens), device=x.device).unsqueeze(0)
            tgt = model.id_embedding(token_tensor) + model.pos_embedding(positions)
            causal_mask = torch.triu(
                torch.ones(len(full_tokens), len(full_tokens), device=x.device), diagonal=1
            ).bool()
            output = model.decoder(tgt, memory, tgt_mask=causal_mask)
            logits = model.output_proj(output[:, -1]).squeeze(0)
            
            # Mask invalid tokens
            mask = torch.full_like(logits, -1e9)
            for vt in valid_tokens:
                mask[vt] = 0
            log_probs = F.log_softmax(logits + mask, dim=-1)
            
            for vt in valid_tokens:
                new_score = score + log_probs[vt].item()
                all_candidates.append((new_score, tokens + [vt]))
        
        if not all_candidates:
            break
        
        beams = sorted(all_candidates, key=lambda x: -x[0])[:beam_size]
    
    return beams

# Test constrained beam search
test_user = 0
hist = user_sequences[test_user][:-1][-20:]
target = user_sequences[test_user][-1]
hist_feat = torch.tensor(item_features[hist].mean(axis=0), dtype=torch.float).unsqueeze(0)

results = constrained_beam_search(model, hist_feat, trie, beam_size=10)

print(f"Target item: {target} (category: {item_categories[target]})")
print(f"Target semantic ID: {tuple(semantic_ids[target].tolist())}")
print(f"\nBeam search results:")
for score, tokens in results[:5]:
    items = trie.lookup(tuple(tokens))
    cats = [item_categories[i] for i in items[:3]]
    print(f"  Score: {score:.3f} | ID: ({tokens[0]}, {tokens[1]}, {tokens[2]}) | "
          f"Items: {items[:3]} | Categories: {cats}")

## 5. TDM and Deep Retrieval

### TDM: Tree-based Deep Model (Alibaba, 2018)
TDM organizes items in a tree structure and uses beam search over the tree:
- Root → Level 1 nodes → Level 2 nodes → ... → Leaf items
- At each level, score and prune candidate nodes
- The tree is learned jointly with the model

### DR: Deep Retrieval (Google, 2020)
DR uses a multi-path structure:
- Items are assigned to multiple paths (unlike single-path in tree models)
- EM algorithm: E-step assigns items to paths, M-step trains the model
- Captures the fact that items can belong to multiple categories

> **💡 Concept:** The key difference between TDM/DR and TIGER is the indexing structure. TDM uses a learned tree, DR uses multi-path assignments, and TIGER uses RQ-VAE codes. All achieve hierarchical retrieval without an external ANN index.

In [ ]:
class SimpleTDM(nn.Module):
    """Simplified Tree-based Deep Model.
    
    Uses a balanced binary tree for item organization.
    """
    def __init__(self, num_items: int, embed_dim: int = 32, tree_depth: int = 4):
        super().__init__()
        self.tree_depth = tree_depth
        self.num_leaves = 2 ** tree_depth
        self.num_nodes = 2 ** (tree_depth + 1) - 1
        
        # Node embeddings for the tree
        self.node_embedding = nn.Embedding(self.num_nodes, embed_dim)
        
        # User encoder
        self.user_proj = nn.Linear(embed_dim, embed_dim)
        
        # Scoring function
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # Item to leaf assignment (simplified: random assignment)
        self.item_to_leaf = np.random.randint(0, self.num_leaves, num_items)
    
    def get_path(self, leaf_idx: int) -> List[int]:
        """Get path from root to leaf (node indices)."""
        path = []
        node = leaf_idx + (self.num_leaves - 1)  # Offset to leaf nodes
        while node >= 0:
            path.append(node)
            node = (node - 1) // 2 if node > 0 else -1
        return path[::-1]  # Root to leaf
    
    def score_nodes(self, user_emb: torch.Tensor, node_ids: torch.Tensor) -> torch.Tensor:
        """Score user-node pairs."""
        user_h = self.user_proj(user_emb)  # (B, D)
        node_h = self.node_embedding(node_ids)  # (B, N, D) or (B, D)
        
        if node_h.dim() == 2:
            combined = torch.cat([user_h, node_h], dim=-1)
        else:
            user_h_expanded = user_h.unsqueeze(1).expand_as(node_h)
            combined = torch.cat([user_h_expanded, node_h], dim=-1)
        
        return self.scorer(combined).squeeze(-1)
    
    def beam_search_retrieve(self, user_emb: torch.Tensor, beam_size: int = 10):
        """Retrieve items by beam search over the tree."""
        B = user_emb.size(0)
        
        # Start from root
        current_nodes = torch.zeros(B, 1, dtype=torch.long)  # root = 0
        
        for depth in range(self.tree_depth):
            # Expand to children
            left_children = current_nodes * 2 + 1
            right_children = current_nodes * 2 + 2
            children = torch.cat([left_children, right_children], dim=1)  # (B, 2*beam)
            
            # Clamp to valid range
            children = children.clamp(max=self.num_nodes - 1)
            
            # Score children
            scores = self.score_nodes(user_emb, children)  # (B, 2*beam)
            
            # Keep top-beam_size
            k = min(beam_size, children.size(1))
            top_scores, top_indices = scores.topk(k, dim=1)
            current_nodes = children.gather(1, top_indices)  # (B, beam)
        
        # Map leaf nodes to items
        leaf_indices = current_nodes - (self.num_leaves - 1)
        return leaf_indices.clamp(min=0)

tdm = SimpleTDM(NUM_ITEMS, embed_dim=EMBED_DIM, tree_depth=4)
user_emb = torch.randn(4, EMBED_DIM)
retrieved_leaves = tdm.beam_search_retrieve(user_emb, beam_size=5)
print(f"TDM retrieved leaf indices: {retrieved_leaves.shape}")
print(f"Tree: {tdm.num_nodes} nodes, {tdm.num_leaves} leaves, depth {tdm.tree_depth}")

## 6. Comparison: Index-Based vs Model-Based Retrieval

In [ ]:
# Comparison table visualization
comparison_data = {
    'Method': ['Two-Tower+ANN', 'DSI', 'TIGER', 'TDM', 'Deep Retrieval'],
    'Type': ['Index-based', 'Model-based', 'Model-based', 'Model-based', 'Model-based'],
    'Index': ['ANN (Faiss/HNSW)', 'Model weights', 'Model weights', 'Learned tree', 'Multi-path'],
    'Update': ['Re-index', 'Re-train', 'Re-train', 'Re-train+tree', 'EM + retrain'],
    'Scalability': ['Billions', 'Millions', 'Millions', '100M+', 'Millions'],
    'Cold Start': ['No', 'No', 'Partial', 'No', 'No'],
}

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

table = ax.table(
    cellText=[comparison_data[k] for k in comparison_data],
    rowLabels=list(comparison_data.keys()),
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)

# Color header row
for j in range(len(comparison_data['Method'])):
    table[(0, j)].set_facecolor('#3498db')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

plt.title('Index-Based vs Model-Based Retrieval Comparison', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement Multi-Target Generative Retrieval

Modify the generative model to retrieve multiple items by generating a sequence of semantic IDs separated by a special separator token.

In [ ]:
class MultiTargetDSI(nn.Module):
    """Generate multiple item IDs for a single query."""
    def __init__(self, input_dim, vocab_size, num_id_tokens, d_model=64,
                 max_items=5):
        super().__init__()
        # TODO: Add separator token to vocabulary
        # TODO: Modify decoder to generate: id1 SEP id2 SEP id3 ...
        # TODO: During beam search, track completed items
        pass
    
    def forward(self, x, target_id_sequences):
        # TODO: Teacher-forced training on concatenated ID sequences
        pass
    
    def generate(self, x, num_items=5, beam_size=10):
        # TODO: Generate until num_items are produced or max length reached
        pass

### 🏋️ Exercise 2: Add Indexing Task to DSI Training

DSI trains on both indexing (document → ID) and retrieval (query → ID). Implement multi-task training.

In [ ]:
def train_dsi_multitask(model, train_loader, item_features, semantic_ids,
                         num_epochs=20, index_weight=0.5):
    """
    Train DSI with both indexing and retrieval tasks.
    
    Indexing: item_features -> semantic_id
    Retrieval: user_features -> semantic_id
    
    Loss = (1-w) * retrieval_loss + w * indexing_loss
    """
    # TODO: In each batch, alternate between indexing and retrieval tasks
    # TODO: For indexing: sample random items, predict their semantic IDs
    # TODO: For retrieval: use user history features, predict next item's semantic ID
    # TODO: Combine losses with weighting
    pass

### 🏋️ Exercise 3: Build End-to-End TIGER Pipeline

Combine RQ-VAE semantic ID assignment with generative retrieval in a unified pipeline.

In [ ]:
def tiger_pipeline(item_features, user_sequences, 
                    num_codes=32, num_levels=3, embed_dim=32):
    """
    Full TIGER pipeline:
    1. Train RQ-VAE to assign semantic IDs to all items
    2. Build trie of valid semantic IDs
    3. Train generative retrieval model
    4. Evaluate with constrained beam search
    
    Returns:
        rqvae_model, retrieval_model, trie, evaluation_metrics
    """
    # TODO: Step 1 - Train RQ-VAE
    # TODO: Step 2 - Build semantic ID trie
    # TODO: Step 3 - Train generative retrieval model
    # TODO: Step 4 - Evaluate with constrained beam search
    # TODO: Report Recall@K and category match rate
    pass

## Summary

Generative retrieval represents a paradigm shift from "search in an index" to "generate the answer directly":

| Model | Year | Company | Key Innovation |
|-------|------|---------|----------------|
| **TDM** | 2018 | Alibaba | Tree-structured retrieval |
| **DR** | 2020 | Google | Multi-path EM |
| **DSI** | 2022 | Google | Document-as-token paradigm |
| **GENRE** | 2021 | Meta | Constrained beam search for entities |
| **NCI** | 2022 | Various | Prefix-aware encoding |
| **TIGER** | 2023 | Google | RQ-VAE + autoregressive = best semantic IDs |

**Current status**: Generative retrieval is promising but still behind two-tower+ANN at billion scale. Its strengths are in avoiding index maintenance and enabling end-to-end optimization.

**Next up**: Chapter 3.8 covers Pre-ranking and Lightweight Scoring — the bridge between retrieval and full ranking.